In [9]:
from pktest import *
from asprefix import *
import json
import re
import numpy as np
from tqdm import tqdm

In [2]:
asip = {}
with open('all_targets.jsonl') as f:
    for line in f:
        a = json.loads(line)
        asip[a[0]] = a[1]
        # break
# with open('asip.jsonl') as f:
#     for line in f:
#         a = json.loads(line)
#         del asip[a[0]]

In [4]:
np.max([len(x) for x in asip.values()])

948798

In [11]:
asip = {}
with open('out_rt_1.jsonl') as f:
    for line in f:
        a = json.loads(line)
        if len(a[1]) > 0 and a[0] not in asip:
            asip[a[0]] = (1, a[1][0])
with open('out_ping_1.jsonl') as f:
    for line in f:
        a = json.loads(line)
        if len(a[1]) > 0 and a[0] not in asip:
            asip[a[0]] = (0, a[1][0])
with open('out_ping_2.jsonl') as f:
    for line in f:
        a = json.loads(line)
        if len(a[1]) > 0 and a[0] not in asip:
            asip[a[0]] = (0, a[1][0])
with open('all_targets.jsonl') as f:
    for line in f:
        a = json.loads(line)
        if len(a[1]) > 0 and a[0] not in asip:
            asip[a[0]] = (-1, a[1][0])
for i in as_prefix:
    if i not in asip:
        asip[i] = (-2, as_prefix[i][0].split('/')[0])
with open('asip.jsonl', 'w') as f:
    for k, v in asip.items():
        f.write(json.dumps([k, *v]) + '\n')
len(asip)

KeyError: 20542

In [2]:
asip = {}
with open('targets.jsonl') as f:
    for line in f:
        a = json.loads(line)
        if a[0] in tier1:
            asip[a[0]] = a[1]
len(asip), len(tier1)

(16, 16)

In [5]:
r = TestRunner()
r.add_tests(TestGroupASIP.make_groups(asip, ttl_min=10, ttl_max=20, max_fail=3))
with tqdm(total=r.count_remain(), unit='tests') as pbar:
    while not r.done():
        pbar.update(r.run_batch())

100%|██████████| 16/16 [03:42<00:00, 13.90s/tests]


In [6]:
asip1 = r.get_results()
asip1

[(174, ['154.39.99.1']),
 (701, []),
 (1299, ['213.248.83.37']),
 (2914, ['165.254.163.1']),
 (3257, ['63.141.214.1']),
 (3320, ['217.239.55.210']),
 (3356, ['8.21.164.108']),
 (3491, ['63.220.193.207']),
 (5511, ['193.251.142.88']),
 (6453, ['64.86.160.2']),
 (6461, ['209.183.189.254']),
 (6762, ['5.178.43.227']),
 (6830, ['89.101.166.6']),
 (7018, []),
 (7922, ['50.237.116.1']),
 (12956, ['94.142.99.151'])]

In [14]:
r1 = TestRunner()
r1.add_tests(TestGroupTTL.make_groups(['157.130.228.150', '12.55.112.242'], 'icmp', 'all', ttl_min=5, ttl_max=30))
with tqdm(total=r1.count_remain(), unit='tests') as pbar:
    while not r1.done():
        pbar.update(r1.run_batch())

100%|██████████| 7/7 [01:09<00:00,  9.93s/tests]


In [15]:
r1.get_results()

[(('157.130.228.150', 'icmp', 'none'),
  ['36.110.203.229',
   '36.110.246.1',
   '202.97.61.158',
   '202.97.55.214',
   '202.97.59.110']),
 (('157.130.228.150', 'icmp', 'rr'),
  ['36.110.203.233',
   '36.110.245.169',
   '202.97.61.158',
   '202.97.54.14',
   '202.97.41.198']),
 (('157.130.228.150', 'icmp', 'nop'),
  ['36.110.203.233',
   '36.110.245.169',
   '202.97.61.158',
   '202.97.54.14',
   '202.97.41.198']),
 (('157.130.228.150', 'icmp', 'unkctl3'),
  ['36.110.203.233',
   '36.110.245.169',
   '202.97.61.158',
   '202.97.54.14',
   '202.97.41.198']),
 (('157.130.228.150', 'icmp', 'unkres3'),
  ['36.110.203.233',
   '36.110.245.169',
   '202.97.61.158',
   '202.97.54.14',
   '202.97.41.198']),
 (('157.130.228.150', 'icmp', 'unkctl40'),
  ['36.110.203.233',
   '36.110.245.169',
   '202.97.61.158',
   '202.97.54.14',
   '202.97.41.198']),
 (('157.130.228.150', 'icmp', 'unkres40'),
  ['36.110.203.233',
   '36.110.245.169',
   '202.97.61.158',
   '202.97.54.14',
   '202.97.41.198'

In [12]:
paths = r1.get_results()
for (ip, l4, opt), path in paths:
    asn = get_ip_asn(ip)[1]
    last = path[-1]
    lastasn = get_ip_asn(last)
    print(ip, last, asn, l4, opt, ip == last, lastasn and asn == lastasn[1])

62.115.138.169 62.115.138.169 1299 icmp none True True
62.115.138.169 62.115.113.21 1299 icmp rr False True
62.115.138.169 62.115.138.169 1299 icmp nop True True
62.115.138.169 62.115.138.169 1299 icmp unkctl3 True True
62.115.138.169 62.115.138.169 1299 icmp unkres3 True True
62.115.138.169 62.115.138.169 1299 icmp unkctl40 True True
62.115.138.169 62.115.138.169 1299 icmp unkres40 True True
208.116.131.138 208.116.131.138 3257 icmp none True True
208.116.131.138 208.116.131.138 3257 icmp rr True True
208.116.131.138 208.116.131.138 3257 icmp nop True True
208.116.131.138 208.116.131.138 3257 icmp unkctl3 True True
208.116.131.138 208.116.131.138 3257 icmp unkres3 True True
208.116.131.138 208.116.131.138 3257 icmp unkctl40 True True
208.116.131.138 208.116.131.138 3257 icmp unkres40 True True
129.250.2.146 129.250.2.146 2914 icmp none True True
129.250.2.146 129.250.2.146 2914 icmp rr True True
129.250.2.146 129.250.2.146 2914 icmp nop True True
129.250.2.146 129.250.2.146 2914 icmp 

In [6]:
stats = {}
any = set()
for i in r.get_results():
    if not i[1] or i[1][-1] != i[0][0]:
        continue
    stats.setdefault(i[0][1:], 0)
    stats[i[0][1:]] += 1
    if i[0][2] != 'none':
        any.add(i[0][0])
stats['any'] = len(any)
stats

{('icmp', 'none'): 844,
 ('icmp', 'nop'): 539,
 ('icmp', 'unkctl3'): 543,
 ('icmp', 'unkres3'): 544,
 ('icmp', 'unkctl40'): 540,
 ('icmp', 'unkres40'): 545,
 ('icmp', 'rr'): 350,
 'any': 556}